# Training on Colab

1. Locally: `uv run python scripts/prepare_colab.py`, then upload `data/colab/doc_verify_bundle.zip` to Google Drive as `MyDrive/doc_verify/doc_verify_bundle.zip`.
2. Here: *Runtime → Change runtime type → T4 GPU*, then run the cells in order.
3. Every run writes to `MyDrive/doc_verify/runs/<name>` and is skipped if it already finished, so a disconnect only costs the run in progress.

Only the train and val splits are used; the test split is locked in `doc_verify/config.py`.

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/doc_verify'
ROOT = '/content/bundle'
!mkdir -p {ROOT} && unzip -q -o {DRIVE}/doc_verify_bundle.zip -d {ROOT}
!pip -q install timm

sys.path.insert(0, f'{ROOT}/code')
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - switch the runtime to T4')

In [ ]:
# frozen CLIP text encoder, run once: per-token outputs of 8 overlapping windows per page, fp16, kept on Drive
from doc_verify.data import all_pages, source_dir
from doc_verify.windows import TEXT_WINDOWS, cache_text_features

CACHE = '/content/clip_text'
DRIVE_CACHE = f'{DRIVE}/cache/clip_text'
if not os.path.exists(f'{DRIVE_CACHE}_features.npy'):
    pages = all_pages(ROOT)
    texts = [(source_dir(p.source, ROOT) / p.text_path).read_text(encoding='utf-8') for p in pages.itertuples()]
    cache_text_features(texts, CACHE, TEXT_WINDOWS)
    os.makedirs(f'{DRIVE}/cache', exist_ok=True)
    for part in ('features', 'mask'):
        shutil.copy(f'{CACHE}_{part}.npy', f'{DRIVE_CACHE}_{part}.npy')
else:
    for part in ('features', 'mask'):
        shutil.copy(f'{DRIVE_CACHE}_{part}.npy', f'{CACHE}_{part}.npy')
print('cache ready')

In [ ]:
from doc_verify.train import TrainConfig, train

RUNS = {
    'fused': TrainConfig(model='fused'),
    'fused_no_kind': TrainConfig(model='fused', lambda_kind=0),
    'image_vit': TrainConfig(model='image_vit'),
    'resnet18': TrainConfig(model='resnet18', lambda_auth=0, lambda_kind=0, lr_backbone=1e-3, lr_head=1e-3,
                            epochs=30, patience=5),
    # diagnostic: SAD-Syn only, no type loss, best epoch chosen by validation authenticity AUC alone
    'fused_auth_only': TrainConfig(model='fused', use_type=False, epochs=40, patience=10),
    # same, without early stopping: does longer training help, or does it only memorise?
    'fused_auth_long': TrainConfig(model='fused', use_type=False, epochs=40, patience=40),
}


def run(name):
    out = f'{DRIVE}/runs/{name}'
    if os.path.exists(f'{out}/best_metrics.json'):
        print(name, 'already finished')
        return
    train(RUNS[name], out, data_root=ROOT, text_cache=CACHE)

In [ ]:
run('fused')

In [ ]:
run('fused_no_kind')

In [ ]:
run('image_vit')

In [ ]:
run('resnet18')

In [ ]:
run('fused_auth_only')

In [ ]:
run('fused_auth_long')

In [ ]:
metrics = {name: Path(f'{DRIVE}/runs/{name}/best_metrics.json') for name in RUNS}
summary = pd.DataFrame({name: json.loads(path.read_text()) for name, path in metrics.items() if path.exists()}).T
summary[[c for c in summary if c.startswith('val_') or c == 'epoch']].round(3)

## Checks before the test set is unlocked (validation only)

Cached text features belong to their pages; saved predictions carry the right labels; authenticity AUC by kind, document type and evidence readability (95% pair-bootstrap intervals); decision thresholds chosen on validation.

In [ ]:
from doc_verify.checks import cache_check, predictions_check
from doc_verify.data import all_pages
from doc_verify.evaluate import auth_breakdown, choose_threshold

pages = all_pages(ROOT)
print('text cache:', cache_check(pages, ROOT, CACHE))

tables, thresholds = [], {}
for name in ['fused', 'fused_no_kind', 'image_vit', 'resnet18', 'fused_auth_only', 'fused_auth_long']:
    preds = pd.read_csv(f'{DRIVE}/runs/{name}/val_predictions.csv')
    print(name, predictions_check(preds, pages))
    authed = preds[preds.auth_label != -1]
    if len(authed):
        tables.append(auth_breakdown(preds, pages).assign(model=name))
        thresholds[name] = choose_threshold(authed.auth_label, authed.auth_score)

os.makedirs(f'{DRIVE}/checks', exist_ok=True)
breakdown = pd.concat(tables)
breakdown.to_csv(f'{DRIVE}/checks/val_auth_breakdown.csv', index=False)
Path(f'{DRIVE}/checks/thresholds.json').write_text(json.dumps(thresholds, indent=2))
print(thresholds)
breakdown.pivot_table(index=['group', 'value'], columns='model', values='auc', sort=False).round(3)